In [ ]:
import stim

import sinter

from stimbposd import BPOSD, sinter_decoders

import numpy as np

import matplotlib.pyplot as plt

import multiprocessing

from pathlib import Path

In [ ]:
import sys
from pathlib import Path

if str(Path.cwd().parent) not in sys.path:
    sys.path.insert(0, str(Path.cwd().parent))

from non_circuit_library import reuse as _circuit_library

_circuit_library.configure(3)

from non_circuit_library.reuse import (
    stabilizers_k_z,
    stabilizers_k_x,
    V_even,
    V_odd,
    S_gate_track,
    stabilizers_s_x,
    stabilizers_s_z,
    stabilizers_added_X,
    stabilizers_k_z_enlargable,
    stabilizers_k_z_enlargable_horizontal,
    stabilizers_k_z_enlargable_vertical,
    stabilizers_k_z_enlarged,
    stabilizers_s_z_enlarged,
    stabilizers_added_Z,
    stabilizer_global,
    stabilizers_k_x_enlargable,
    stabilizers_k_x_enlarged,
    stabilizers_s_x_enlargable,
    stabilizers_s_x_enlarged,
    stabilizers_general,
    stabilizers_k_z_unchange,
    stabilizers_merged,
    stabilizers_merged_without_add_X,
    stabilizers_k_x_unchange,
    stabilizers_s_x_unchange,
    stabilizers_merged_string,
    stabilizers_merged_string_without_add_Z,
    lattice_surgery_Merge,
    lattice_surgery_Split,
    lattice_surgery_Merge_string,
    lattice_surgery_Split_string,
    measure_logical_qubits_3D,
    measure_logical_qubits_2D,
    measurement_XX,
    measurement_ZZ,
    S_Z_Gate,
    S_Z_DAG_Gate,
    logical_CZ,
    DISTANCE,
    Stabilizers_measurement_general,
    Stabilizers_measurement_general_after_gate,
    Stabilizers_measurement_merged,
    Stabilizers_measurement_merged_string,
    detector_k_x,
)


In [ ]:
stabilizers_general

In [ ]:
def circuit_generate(rate_mea, rate_idle):

    qubit_range_3D = list(range(0, 15))    
    qubit_range_2D = list(range(21, 28))

    qubit_range_data = qubit_range_3D + qubit_range_2D
    qubit_range_all = list(range(0, 28))
    
    Observable_list_XX = []
    Observable_list_ZZ = []
    
    c = stim.Circuit(); c.append("H", qubit_range_3D)


    c += Stabilizers_measurement_general(0, 1)
    c.append("TICK")

# H on 2D code: +0
    c.append("H", qubit_range_2D)
    c.append("DEPOLARIZE1", qubit_range_data, rate_idle) 
    c += Stabilizers_measurement_general_after_gate(rate_mea, 1, After_H_2D=True)
    c.append("TICK") 

# logical CZ: 00 + 11
    c += logical_CZ()
    c.append("DEPOLARIZE1", qubit_range_data, rate_idle) 
    c += Stabilizers_measurement_general_after_gate(rate_mea, 2, After_CZ=True)
    c.append("TICK")

    
# Lattice surgery to measure ZZ

    # Merging_ZZ
    
    c += lattice_surgery_Merge_string()
    
    c.append("DEPOLARIZE1", qubit_range_all, rate_idle) 
    c += Stabilizers_measurement_merged_string(rate_mea, 4, First_merging_round = True) 
    c.append("TICK")

    c.append("DEPOLARIZE1", qubit_range_all, rate_idle) 
    c += Stabilizers_measurement_merged_string(rate_mea, 5)
    c.append("TICK")

    c.append("DEPOLARIZE1", qubit_range_all, rate_idle) 
    c += Stabilizers_measurement_merged_string(rate_mea, 6, offset=c.num_measurements, Observable_list_XX=Observable_list_XX) 
    c.append("TICK")
    

    
    # Splitting_ZZ

    c += lattice_surgery_Split_string(rate_mea)
    
    c.append("DEPOLARIZE1", qubit_range_data, rate_idle) 
    c += Stabilizers_measurement_general(rate_mea, 7, After_merging_string=True)
    c.append("TICK")
    
    c.append("DEPOLARIZE1", qubit_range_data, rate_idle) 
    c += Stabilizers_measurement_general(rate_mea, 8)
    c.append("TICK")

    c.append("DEPOLARIZE1", qubit_range_data, rate_idle) 
    c += Stabilizers_measurement_general(rate_mea, 9)
    c.append("TICK")
    

    c.append("R", [15, 16])
    




# S gate on 3D code
    c += S_Z_Gate()
    c.append("DEPOLARIZE1", qubit_range_data, rate_idle) 
    c += Stabilizers_measurement_general_after_gate(rate_mea, 11, After_S_3D=True)


    
# H + Face-to-Face XX lattice surgery + H measures ZX

    
    # Merging_XX

# H on 2D code: before the Face-to-Face XX measurement
    c.append("H", qubit_range_2D)
    c.append("DEPOLARIZE1", qubit_range_data, rate_idle) 
    c += Stabilizers_measurement_general_after_gate(rate_mea, 12, After_H_2D=True)
    c.append("TICK") 
    c += lattice_surgery_Merge()
    
    c.append("DEPOLARIZE1", qubit_range_all, rate_idle) 
    c += Stabilizers_measurement_merged(rate_mea, 13, First_merging_round=True) 
    c.append("TICK")

    c.append("DEPOLARIZE1", qubit_range_all, rate_idle) 
    c += Stabilizers_measurement_merged(rate_mea, 14) 
    c.append("TICK")

    c.append("DEPOLARIZE1", qubit_range_all, rate_idle) 
    c += Stabilizers_measurement_merged(rate_mea, 15, offset=c.num_measurements, Observable_list_ZZ=Observable_list_ZZ) 
    c.append("TICK")



    
    # Splitting_XX

    c += lattice_surgery_Split(rate_mea)
    
    c.append("DEPOLARIZE1", qubit_range_data, rate_idle) 
    c += Stabilizers_measurement_general(rate_mea, 16, After_merging=True)
    c.append("TICK")
    
    c.append("DEPOLARIZE1", qubit_range_data, rate_idle) 
    c += Stabilizers_measurement_general(rate_mea, 17)
    c.append("TICK")

    c.append("DEPOLARIZE1", qubit_range_data, rate_idle) 
    c += Stabilizers_measurement_general(rate_mea, 18)
    c.append("TICK")

# H on 2D code: after the Face-to-Face XX measurement
    c.append("H", qubit_range_2D)
    c.append("DEPOLARIZE1", qubit_range_data, rate_idle) 
    c += Stabilizers_measurement_general_after_gate(rate_mea, 19, After_H_2D=True)
    c.append("TICK")
    c += logical_CZ()
    c.append("DEPOLARIZE1", qubit_range_data, rate_idle) 
    c += Stabilizers_measurement_general_after_gate(rate_mea, 20, After_CZ=True)
    c.append("TICK")

    c += Stabilizers_measurement_general(0, 21)
    c.append("TICK")


# set the determinstic observable
    
    c.append("MPP", stim.PauliString("Y21*Y22*Y23*Y24*Y25*Y26*Y27"), tag="logical_qubits_2D") 
    # c.append("MPP", stim.PauliString("Y0*Y3*Y6*Y7*Y10*Y13*Y14"), tag="logical_qubits_3D")
    
    observable_targets_XX = []
    observable_targets_ZZ = []

    # Convert all the X_recs indices:
    for absolute_index in Observable_list_XX:
        observable_targets_XX.append(stim.target_rec(absolute_index-c.num_measurements))
    for absolute_index in Observable_list_ZZ:
        observable_targets_ZZ.append(stim.target_rec(absolute_index-c.num_measurements))


    
    # c.append("OBSERVABLE_INCLUDE", observable_targets_XX, 0)
    # c.append("OBSERVABLE_INCLUDE", observable_targets_ZZ, 1)

    
    c.append("OBSERVABLE_INCLUDE", [stim.target_rec(-1), *observable_targets_XX, *observable_targets_ZZ], 0)


    return c

In [ ]:
c = circuit_generate(0.01, 0.01)
# c.diagram("timeline-svg")

In [ ]:
dem = c.detector_error_model()

In [ ]:
len(c.shortest_graphlike_error())

In [ ]:
def generate_tasks():
    for p in [0.002, 0.004, 0.006, 0.008, 0.01]:
            yield sinter.Task(
                circuit=circuit_generate(p, p),
                json_metadata={'p': p},
            )

In [ ]:
samples = sinter.collect(
    num_workers=multiprocessing.cpu_count() - 1,
    max_shots=1000_000,
    max_errors=1000,
    tasks=generate_tasks(),
    decoders=["hypergraph_union_find"],
    custom_decoders=sinter_decoders(),
    save_resume_filepath="Logical_T_d3.csv"
)

In [ ]:
fig, ax = plt.subplots(1, 1)
sinter.plot_error_rate(  
    ax=ax,  
    stats=samples,  
    group_func=lambda stat: stat.decoder,  # No 'd' available  
    x_func=lambda stat: stat.json_metadata['p']  # Placeholder x since 'p' is unavailable  
)
ax.loglog()
ax.grid()
ax.set_title("Logical Error Rate vs Physical Error Rate")
ax.set_ylabel("Logical Error Probability (per shot)")
ax.set_xlabel("Physical Error Rate")
ax.legend()
plt.show()